In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_dublin_core.xml", "data/metadata/climate_dataset_dublin_core.xml"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_dublin_core.xml
Bootstrap complete.


# Day 2: Parse Dublin Core (Offline)

In this notebook we parse a local **Dublin Core** XML metadata record for the teaching dataset “2025 Global Climate Data”.

## Why namespaces matter (brief)
XML uses namespaces to avoid name collisions. In practice, you may see tags like `dc:title` where:
- `dc` is a prefix for a namespace URI
- `title` is the element local name

Our teaching parser matches elements by local name, so students can focus on the meaning rather than XML namespace syntax.

### Colab tip
If `../../data/metadata/...` does not exist in Colab, upload `climate_dataset_dublin_core.xml` and the notebook will use `/content/climate_dataset_dublin_core.xml`.


In [4]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# ---- Notebook-local helper functions (Colab/Jupyter friendly) ----
def _local_name(tag: str) -> str:
    return tag.split('}', 1)[1] if '}' in tag else tag

def _first_text_by_local_name(root: ET.Element, target: str):
    for el in root.iter():
        if _local_name(el.tag) == target:
            txt = (el.text or '').strip()
            if txt:
                return txt
    return None

def _texts_by_local_name(root: ET.Element, target: str):
    out = []
    for el in root.iter():
        if _local_name(el.tag) == target:
            txt = (el.text or '').strip()
            if txt:
                out.append(txt)
    return out

def parse_dublin_core_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    return {
        'standard': 'Dublin Core',
        'title': _first_text_by_local_name(root, 'title'),
        'creator': _texts_by_local_name(root, 'creator'),
        'publisher': _first_text_by_local_name(root, 'publisher'),
        'identifier': _first_text_by_local_name(root, 'identifier'),
        'description': _first_text_by_local_name(root, 'description'),
        'subject': _texts_by_local_name(root, 'subject'),
        'date': _first_text_by_local_name(root, 'date'),
        'format': _first_text_by_local_name(root, 'format'),
    }

# Path strategy:
# - In this repository: use ../../data/metadata/...
# - In Colab: upload the metadata file and point to /content/<filename>
default_path = Path('/content/data/metadata/climate_dataset_dublin_core.xml')
dc_path = default_path if default_path.exists() else Path('/content/climate_dataset_dublin_core.xml')

dc_meta = parse_dublin_core_xml(dc_path)
dc_meta

{'standard': 'Dublin Core',
 'title': '2025 Global Climate Data',
 'creator': ['Global Climate Data Team'],
 'publisher': 'Open Research Repository (Teaching)',
 'identifier': '10.1234/gcd-2025',
 'description': 'A small teaching dataset containing fictional annual climate summaries.\n    Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.',
 'subject': ['climate', 'temperature', 'rainfall', 'CO2 emissions'],
 'date': '2025-12-31',
 'format': 'text/csv'}

In [5]:
# Convert to a small table for readability
table = pd.DataFrame(
    [
        {"field": "title", "value": dc_meta.get("title")},
        {"field": "creator", "value": "; ".join(dc_meta.get("creator") or [])},
        {"field": "publisher", "value": dc_meta.get("publisher")},
        {"field": "identifier", "value": dc_meta.get("identifier")},
        {"field": "description", "value": dc_meta.get("description")},
        {"field": "subject", "value": "; ".join(dc_meta.get("subject") or [])},
        {"field": "date", "value": dc_meta.get("date")},
        {"field": "format", "value": dc_meta.get("format")},
    ]
)

table

,field,value
0,title,2025 Global Climate Data
1,creator,Global Climate Data Team
2,publisher,Open Research Repository (Teaching)
3,identifier,10.1234/gcd-2025
4,description,A small teaching dataset containing fictional ...
5,subject,climate; temperature; rainfall; CO2 emissions
6,date,2025-12-31
7,format,text/csv


## Quick interpretation
- The `identifier` field contains the persistent DOI-like identifier.
- The `description` and `subject` fields help interpretation and reuse.
- The `format` field is a starting point for interoperability (machine-readable file type).
